# **Task 1: Build a Relational Schema*
Model an online movie rental service. Design a normalized relational schema using pandas DataFrames for the following entities:

**Step 1: Create these tables:**

- customers with columns: customer_id, name, email, city, signup_date
- movies with columns: movie_id, title, genre, release_year, rating (1-5 scale)
- rentals with columns: rental_id, customer_id, movie_id, rental_date, return_date, price

Populate with at least 8 customers, 10 movies, and 25 rentals. Make sure some customers have multiple rentals and some movies are rented multiple times.

In [44]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [45]:
random.seed(42)

In [46]:
# 1. Customers (Fixed the date addition error)
customer_count = 8
df_customers = pd.DataFrame({
    'customer_id': range(1, customer_count + 1),
    'name': [f'Customer_{i}' for i in range(1, customer_count + 1)],
    'email': [f'user{i}@example.com' for i in range(1, customer_count + 1)],
    'city': random.choices(['New York', 'Chicago', 'Austin', 'Miami'], k=customer_count),
    # We create a series of the same date repeated 8 times so the math works
    'signup_date': pd.to_datetime(['2025-01-01'] * customer_count) + pd.to_timedelta(np.random.randint(0, 60, size=customer_count), unit='D')
})

# 2. Movies (10 movies)
movie_count = 10
df_movies = pd.DataFrame({
    'movie_id': range(101, 101 + movie_count),
    'title': [f'Movie_Title_{i}' for i in range(1, movie_count + 1)],
    'genre': random.choices(['Sci-Fi', 'Action', 'Comedy', 'Drama', 'Horror'], k=movie_count),
    'release_year': np.random.randint(1970, 2025, size=movie_count),
    'rating': np.random.randint(1, 6, size=movie_count)
})

# 3. Rentals (25 rentals)
rental_count = 25
df_rentals = pd.DataFrame({
    'rental_id': range(1, rental_count + 1),
    'customer_id': np.random.randint(1, 9, size=rental_count),
    'movie_id': np.random.randint(101, 111, size=rental_count),
    'price': np.round(np.random.uniform(1.99, 5.99, size=rental_count), 2)
})

print("Success! DataFrames are created with randomized values.")

Success! DataFrames are created with randomized values.


**Step 2: Write and execute these queries using pandas operations:**

1. Find all movies rented by a specific customer (requires joining rentals with movies).
2. Find the top 5 most-rented movies (group and count).
3. Compute the total revenue per genre (join rentals with movies, group by genre, sum price).
4. Find customers who have never rented a movie rated below 3 (multi-step filtering).

In [47]:
df_customers

,customer_id,name,email,city,signup_date
0,1,Customer_1,user1@example.com,Austin,2025-01-13
1,2,Customer_2,user2@example.com,New York,2025-02-23
2,3,Customer_3,user3@example.com,Chicago,2025-01-31
3,4,Customer_4,user4@example.com,New York,2025-01-22
4,5,Customer_5,user5@example.com,Austin,2025-02-25
5,6,Customer_6,user6@example.com,Austin,2025-01-15
6,7,Customer_7,user7@example.com,Miami,2025-02-02
7,8,Customer_8,user8@example.com,New York,2025-02-04


In [48]:
df_movies

,movie_id,title,genre,release_year,rating
0,101,Movie_Title_1,Comedy,1986,4
1,102,Movie_Title_2,Sci-Fi,1991,4
2,103,Movie_Title_3,Action,1993,5
3,104,Movie_Title_4,Comedy,2016,5
4,105,Movie_Title_5,Sci-Fi,2011,3
5,106,Movie_Title_6,Sci-Fi,2006,4
6,107,Movie_Title_7,Drama,1988,4
7,108,Movie_Title_8,Comedy,2001,2
8,109,Movie_Title_9,Action,1984,3
9,110,Movie_Title_10,Comedy,2008,2


In [49]:
df_rentals

,rental_id,customer_id,movie_id,price
0,1,1,102,4.87
1,2,8,105,5.31
2,3,5,106,5.41
3,4,8,106,5.32
4,5,2,107,3.86
5,6,8,106,5.94
6,7,6,102,4.82
7,8,7,103,2.06
8,9,7,106,2.77
9,10,4,109,4.89


In [50]:
# A. Find all movies rented by "Alice Johnson"

In [51]:
# Using the names we defined in the random generation step
alice_rentals = df_rentals[df_rentals['customer_id'] == 1].merge(df_movies, on='movie_id')
alice_rentals[['customer_id', 'title']]

,customer_id,title
0,1,Movie_Title_2
1,1,Movie_Title_7
2,1,Movie_Title_8
3,1,Movie_Title_6
4,1,Movie_Title_8
5,1,Movie_Title_6
6,1,Movie_Title_9


In [52]:
# B.Find the top 5 most-rented movies (group and count).

In [53]:
top_5 = df_rentals.merge(df_movies, on='movie_id') \
    .groupby('title').size() \
    .reset_index(name='count') \
    .sort_values('count', ascending=False)
top_5.head(5)

,title,count
5,Movie_Title_6,7
7,Movie_Title_8,4
2,Movie_Title_2,3
8,Movie_Title_9,3
1,Movie_Title_10,2


In [54]:
# C. Compute the total revenue per genre (join rentals with movies, group by genre, sum price).

In [55]:
revenue = df_rentals.merge(df_movies, on='movie_id') \
    .groupby('genre')['price'].sum() \
    .reset_index() \
    .sort_values('price', ascending=False)
revenue

,genre,price
3,Sci-Fi,52.09
1,Comedy,25.93
0,Action,16.52
2,Drama,7.66


In [56]:
# D. Find customers who have never rented a movie rated below 3 (multi-step filtering).

In [57]:
# Identify customers who rented 'bad' movies
bad_movie_ids = df_movies[df_movies['rating'] < 3]['movie_id']
naughty_list = df_rentals[df_rentals['movie_id'].isin(bad_movie_ids)]['customer_id'].unique()

# Filter them out
high_standards = df_customers[~df_customers['customer_id'].isin(naughty_list)]
high_standards[['name', 'city']]

,name,city
1,Customer_2,New York
4,Customer_5,Austin
6,Customer_7,Miami
7,Customer_8,New York


**Step 3: Demonstrate normalization. Show what the data would look like as a single denormalized table (join all three tables). Write a markdown cell explaining what redundancy appears and why the normalized version is preferable for updates.**

In [58]:
# Join all three DataFrames into one
denormalized_df = df_rentals.merge(df_customers, on='customer_id') \
                            .merge(df_movies, on='movie_id')

# Display the first few rows to see the repetition
denormalized_df.head(10)

,rental_id,customer_id,movie_id,price,name,email,city,signup_date,title,genre,release_year,rating
0,1,1,102,4.87,Customer_1,user1@example.com,Austin,2025-01-13,Movie_Title_2,Sci-Fi,1991,4
1,2,8,105,5.31,Customer_8,user8@example.com,New York,2025-02-04,Movie_Title_5,Sci-Fi,2011,3
2,3,5,106,5.41,Customer_5,user5@example.com,Austin,2025-02-25,Movie_Title_6,Sci-Fi,2006,4
3,4,8,106,5.32,Customer_8,user8@example.com,New York,2025-02-04,Movie_Title_6,Sci-Fi,2006,4
4,5,2,107,3.86,Customer_2,user2@example.com,New York,2025-02-23,Movie_Title_7,Drama,1988,4
5,6,8,106,5.94,Customer_8,user8@example.com,New York,2025-02-04,Movie_Title_6,Sci-Fi,2006,4
6,7,6,102,4.82,Customer_6,user6@example.com,Austin,2025-01-15,Movie_Title_2,Sci-Fi,1991,4
7,8,7,103,2.06,Customer_7,user7@example.com,Miami,2025-02-02,Movie_Title_3,Action,1993,5
8,9,7,106,2.77,Customer_7,user7@example.com,Miami,2025-02-02,Movie_Title_6,Sci-Fi,2006,4
9,10,4,109,4.89,Customer_4,user4@example.com,New York,2025-01-22,Movie_Title_9,Action,1984,3


**Understanding Database Normalization**

**What Redundancy Appears?**

When we join all the tables, we see massive amounts of duplicate data. For every movie Alice Johnson rents:

- Her name, email, city, and signup date are repeated in every row.

- If 50 people rent Inception, the title, genre, and release year for Inception are stored 50 separate times.

**Why Normalization is Preferable for Updates**
Normalization (splitting the data into separate tables) is better for three main reasons:

1. Update Anomalies: If Alice Johnson changes her email address, in a normalized database, we change it in one place (the customers table). In a denormalized table, we would have to search through thousands of rental records and update the email in every single one. If we miss one, the data becomes inconsistent.

2. Storage Efficiency: Text (like names and movie titles) takes up much more space than integers (IDs). Storing "Alice Johnson" once and referring to her as 1 in other tables saves a significant amount of disk space.

3. Insertion/Deletion Issues: If a movie has never been rented yet, a denormalized table might not be able to store the movie's information at all (unless we allow nulls). In a normalized setup, a movie can exist in the movies table regardless of whether it has a rental history.

## **Task 2: Model the Same Data as Documents*
Represent the same movie rental data using the document model.

**Step 1: Create a list of customer documents (Python dictionaries) where each document contains the customer's information and a nested list of their rental history, with movie details embedded in each rental entry.**

In [59]:
# Initialize the list for our document store
customer_documents = []

# Loop through your df_customers DataFrame
for _, customer_row in df_customers.iterrows():
    
    # 1. Create the top-level Customer Document
    doc = {
        "customer_id": int(customer_row['customer_id']),
        "name": customer_row['name'],
        "email": customer_row['email'],
        "city": customer_row['city'],
        "signup_date": customer_row['signup_date'].strftime('%Y-%m-%d'),
        "rentals": [] # List to hold nested rental history
    }
    
    # 2. Filter df_rentals for only this customer's IDs
    customer_rentals = df_rentals[df_rentals['customer_id'] == customer_row['customer_id']]
    
    for _, rental_row in customer_rentals.iterrows():
        # 3. Find the movie details in df_movies for this rental
        movie_details = df_movies[df_movies['movie_id'] == rental_row['movie_id']].iloc[0]
        
        # 4. Create the nested entry (Embedding movie info inside the rental)
        rental_entry = {
            "rental_id": int(rental_row['rental_id']),
            "price": float(rental_row['price']),
            "movie": {
                "movie_id": int(movie_details['movie_id']),
                "title": movie_details['title'],
                "genre": movie_details['genre'],
                "rating": int(movie_details['rating'])
            }
        }
        
        # 5. Append this nested dictionary to the customer's rentals list
        doc["rentals"].append(rental_entry)
        
    # Add the finished document to our collection
    customer_documents.append(doc)

print(f"Successfully modeled {len(customer_documents)} customers as documents.")

Successfully modeled 8 customers as documents.


**Step 2: Execute the same four queries from Task 1 on the document collection:**

1. Find all movies rented by a specific customer.
2. Find the top 5 most-rented movies.
3. Compute total revenue per genre.
4. Find customers who have never rented a movie rated below 3.

In [60]:
target_id = 1
alice_rentals = []

for doc in customer_documents:
    if doc['customer_id'] == target_id:
        for r in doc['rentals']:
            alice_rentals.append({
                "title": r['movie']['title'],
                "genre": r['movie']['genre']
            })
        break

print(f"Movies for Customer {target_id}:")
print(pd.DataFrame(alice_rentals))

Movies for Customer 1:
           title   genre
0  Movie_Title_2  Sci-Fi
1  Movie_Title_7   Drama
2  Movie_Title_8  Comedy
3  Movie_Title_6  Sci-Fi
4  Movie_Title_8  Comedy
5  Movie_Title_6  Sci-Fi
6  Movie_Title_9  Action


In [61]:
from collections import Counter

counts = Counter()
for doc in customer_documents:
    for r in doc['rentals']:
        counts[r['movie']['title']] += 1

print("Top 5 Most-Rented Movies:")
print(counts.most_common(5))

Top 5 Most-Rented Movies:
[('Movie_Title_6', 7), ('Movie_Title_8', 4), ('Movie_Title_2', 3), ('Movie_Title_9', 3), ('Movie_Title_7', 2)]


In [62]:
genre_revenue = {}

for doc in customer_documents:
    for r in doc['rentals']:
        g = r['movie']['genre']
        p = r['price']
        genre_revenue[g] = genre_revenue.get(g, 0) + p

# Convert to DataFrame for a clean view
revenue_df = pd.DataFrame(list(genre_revenue.items()), columns=['Genre', 'Total Revenue'])
print(revenue_df.sort_values('Total Revenue', ascending=False))

    Genre  Total Revenue
0  Sci-Fi          52.09
2  Comedy          25.93
3  Action          16.52
1   Drama           7.66


In [63]:
high_standard_customers = []

for doc in customer_documents:
    # Assume they are high standard until proven otherwise
    has_low_rating = False
    
    for r in doc['rentals']:
        if r['movie']['rating'] < 3:
            has_low_rating = True
            break
            
    if not has_low_rating:
        high_standard_customers.append(doc['name'])

print("Customers with high standards (No rentals below rating 3):")
print(high_standard_customers)

Customers with high standards (No rentals below rating 3):
['Customer_2', 'Customer_5', 'Customer_7', 'Customer_8']


**Step 3: Write a markdown cell comparing the document queries to the relational queries. Which queries were easier to write? Which were harder? For which query pattern does the document model have a clear advantage?**

**Which queries were easier to write?**

- The Relational (Pandas/SQL) queries were generally easier for aggregations (Queries 2 and 3). Using .groupby() and .sum() or .size() allows you to collapse thousands of rows into a summary with just one or two lines of code.

- The Document query was significantly easier for individual lookups (Query 1). Because Alice's movies are already physically stored inside her "folder," we didn't have to perform any complex merges or joins to see her history.

**Which queries were harder?**
- The Document queries were harder for global analysis (Queries 2 and 3). Since the data is "trapped" inside individual customer documents, we had to write "nested loops" (looping through customers, then looping through their rentals) to count movies or sum revenue.

- The Relational queries were harder for initial setup. We had to ensure movie_id matched exactly across three different tables and handle NameErrors if a table wasn't loaded correctly before merging.

**The Clear Advantage of the Document Model**

The document model has a clear advantage for Entity-Centric Access Patterns.

If your application (like Netflix or Amazon) mostly needs to "Load the Profile for User X," the document model is superior. It retrieves the user's name, settings, and full rental history in a single read operation. In a relational model, the database engine has to work much harder to "sew" together information from multiple tables every time a user logs in.

# **Task 3: Model Relationships as a Graph*

Build a graph representation of the rental data to answer relationship-based questions.

**Step 1: Create a graph using Python dictionaries where:**

- Nodes represent customers, movies, and genres.
- Edges represent relationships: rented (customer → movie), belongs_to (movie → genre), lives_in (customer → city).

In [64]:
nodes = {}
edges = []

# 1. Add Customer and City Nodes
for _, row in df_customers.iterrows():
    c_id = f"c{int(row['customer_id'])}"
    city_id = row['city'].lower().replace(" ", "_")
    nodes[c_id] = {"type": "customer", "name": row['name']}
    if city_id not in nodes:
        nodes[city_id] = {"type": "city", "name": row['city']}
    edges.append((c_id, "lives_in", city_id))

# 2. Add Movie and Genre Nodes
for _, row in df_movies.iterrows():
    m_id = f"m{int(row['movie_id'])}"
    g_id = row['genre'].lower()
    nodes[m_id] = {"type": "movie", "title": row['title'], "rating": row['rating']}
    if g_id not in nodes:
        nodes[g_id] = {"type": "genre", "name": row['genre']}
    edges.append((m_id, "belongs_to", g_id))

# 3. Add Rental Edges (CRITICAL FIX: Ensure IDs match prefixes)
for _, row in df_rentals.iterrows():
    c_id = f"c{int(row['customer_id'])}"
    m_id = f"m{int(row['movie_id'])}"
    edges.append((c_id, "rented", m_id))

print(f"Graph Rebuilt: {len(nodes)} Nodes and {len(edges)} Edges.")

Graph Rebuilt: 26 Nodes and 43 Edges.


**Step 2: Write traversal functions to answer:**

1. "Which movies did customer X rent?" — one hop from customer through rented edges.
2. "Which genres does customer X prefer?" — two hops: customer → movies → genres, then count.
3. "Which customers share a genre preference with customer X?" — three hops: customer X → movies → genres → other customers who rented movies in the same genre.
4. "Recommend movies for customer X" — find movies in their preferred genres that they have not yet rented.


In [65]:
# Helper to get all neighbors for a specific node based on relationship type
def get_neighbors(node_id, relationship_type):
    return [edge[2] for edge in edges if edge[0] == node_id and edge[1] == relationship_type]

# 1. Which movies did customer X rent? (One Hop)
def get_rented_movies(customer_id):
    movie_ids = get_neighbors(customer_id, "rented")
    return [nodes[m_id]['title'] for m_id in movie_ids]

# 2. Which genres does customer X prefer? (Two Hops)
def get_preferred_genres(customer_id):
    movie_ids = get_neighbors(customer_id, "rented")
    genres_found = []
    for m_id in movie_ids:
        # One more hop: movie -> genre
        genres_found.extend(get_neighbors(m_id, "belongs_to"))
    
    # Count occurrences
    from collections import Counter
    return Counter(genres_found)

# 3. Which customers share a genre preference with customer X? (Three Hops)
def get_similar_customers(customer_id):
    # Customer -> Movie -> Genre
    my_genres = list(get_preferred_genres(customer_id).keys())
    similar_customers = set()
    
    # Reverse hop: Find all movies in those genres
    for g_id in my_genres:
        # Find movies belonging to this genre (reverse of belongs_to)
        movies_in_genre = [e[0] for e in edges if e[2] == g_id and e[1] == "belongs_to"]
        
        # Find customers who rented those movies (reverse of rented)
        for m_id in movies_in_genre:
            customers = [e[0] for e in edges if e[2] == m_id and e[1] == "rented"]
            similar_customers.update(customers)
            
    # Remove the original customer from the list
    similar_customers.discard(customer_id)
    return [nodes[c_id]['name'] for c_id in similar_customers]

# 4. Recommend movies for customer X (Multi-Hop)
def recommend_movies(customer_id):
    my_rented_ids = set(get_neighbors(customer_id, "rented"))
    my_genres = list(get_preferred_genres(customer_id).keys())
    
    recommendations = set()
    for g_id in my_genres:
        # Find all movies in these genres
        movies_in_genre = [e[0] for e in edges if e[2] == g_id and e[1] == "belongs_to"]
        for m_id in movies_in_genre:
            # If I haven't rented it yet, recommend it!
            if m_id not in my_rented_ids:
                recommendations.add(nodes[m_id]['title'])
                
    return list(recommendations)

In [66]:
# Use the prefixed ID that matches your 'nodes' dictionary keys
test_id = "c1" 

print(f"Results for {nodes[test_id]['name']}:")
print(f"---")
print(f"1. Rented: {get_rented_movies(test_id)}")
print(f"2. Top Genres: {get_preferred_genres(test_id).most_common(1)}")
print(f"3. Similar Customers: {get_similar_customers(test_id)}")
print(f"4. Recommended for you: {recommend_movies(test_id)}")

Results for Customer_1:
---
1. Rented: ['Movie_Title_2', 'Movie_Title_7', 'Movie_Title_8', 'Movie_Title_6', 'Movie_Title_8', 'Movie_Title_6', 'Movie_Title_9']
2. Top Genres: [('sci-fi', 3)]
3. Similar Customers: ['Customer_6', 'Customer_7', 'Customer_5', 'Customer_3', 'Customer_4', 'Customer_8', 'Customer_2']
4. Recommended for you: ['Movie_Title_4', 'Movie_Title_5', 'Movie_Title_10', 'Movie_Title_3', 'Movie_Title_1']


In [67]:
# Check the first 5 node keys to see the format
print("Current Node Keys:", list(nodes.keys())[:5])

# Check the first 5 edges to see the start/end points
print("Current Edges:", edges[:5])

Current Node Keys: ['c1', 'austin', 'c2', 'new_york', 'c3']
Current Edges: [('c1', 'lives_in', 'austin'), ('c2', 'lives_in', 'new_york'), ('c3', 'lives_in', 'chicago'), ('c4', 'lives_in', 'new_york'), ('c5', 'lives_in', 'austin')]


**Step 3: Write a markdown cell explaining why the recommendation query (Task 3, question 4) would be difficult to express in SQL or with documents. What makes the graph model natural for this kind of traversal?**

In this step, we evaluate why the recommendation query—finding movies in preferred genres that a customer hasn't rented yet—is significantly more natural in a graph model compared to relational or document models.

1. **Why it is difficult in SQL (Relational Model)**

To perform this recommendation in a normalized SQL schema, you would need to execute a complex series of operations:

- Multiple Joins: You must join customers to rentals to movies to find the genres the user likes, then join back to movies to find all titles in those genres.

- Set Difference: You then need to perform a EXCEPT or NOT IN subquery to filter out movies the customer has already rented.

- Verbosity: The resulting query is often long, hard to read, and performance can degrade as the number of joins increases.

2. **Why it is difficult with Documents**

While a document model is great for viewing a single customer's profile, it struggles with cross-entity discovery:

- Data Silos: Information about what other movies exist in a genre is often not stored inside a specific customer's document.

- Full Collection Scans: To find recommendations, the application would likely have to fetch the entire "Movies" collection separately or scan through every other customer's document to see what else is available.

- Application Logic: Most of the heavy lifting (filtering and matching) has to be done in the Python/application code rather than the database level.

3. **Why the Graph Model is Natural**
   
Graphs treat relationships as first-class citizens, making "traversal" the primary way to interact with data:

- Pathfinding: A recommendation is simply a "walk" through the network: (Customer) -> [:rented] -> (Movie) -> [:belongs_to] -> (Genre) <- [:belongs_to] - (Recommended Movie).

- Relationship-Centric: The graph doesn't care about "tables"; it only cares about connections. This allows for "n-degree" separations (like finding movies rented by people with similar tastes) to be expressed in a single traversal.

- Efficiency: Instead of scanning entire tables or collections, the graph engine only visits the specific nodes and edges relevant to that user's path, making it highly efficient for real-time recommendation engines.

# **Task 4: OLTP vs OLAP Workload Analysis*
This task asks you to reason about how different database types handle different workloads.

**Step 1: Create a DataFrame representing a log of 5,000 e-commerce transactions with columns: transaction_id, customer_id, product_id, amount, payment_method, timestamp, status (completed/refunded/pending).**

In [68]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)

# Configuration
n_rows = 5000

# Generating data
data = {
    'transaction_id': [f"TXN-{10000 + i}" for i in range(n_rows)],
    'customer_id': [f"CUST-{np.random.randint(100, 1500)}" for _ in range(n_rows)],
    'product_id': [f"PROD-{np.random.randint(10, 500)}" for _ in range(n_rows)],
    'amount': np.round(np.random.uniform(10.0, 500.0, n_rows), 2),
    'payment_method': np.random.choice(['Credit Card', 'PayPal', 'Crypto', 'Debit Card'], n_rows),
    'status': np.random.choice(['completed', 'refunded', 'pending'], n_rows, p=[0.85, 0.05, 0.10]),
    'timestamp': [datetime(2025, 1, 1) + timedelta(minutes=np.random.randint(0, 525600)) for _ in range(n_rows)]
}

df = pd.DataFrame(data)

# Sort by timestamp for a chronological log
df = df.sort_values(by='timestamp').reset_index(drop=True)

print(df.head())

  transaction_id customer_id product_id  amount payment_method     status  \
0      TXN-13450    CUST-641   PROD-364  393.64         PayPal  completed   
1      TXN-11761    CUST-500   PROD-324  230.98         PayPal  completed   
2      TXN-10670    CUST-767   PROD-413   13.48         PayPal    pending   
3      TXN-14561   CUST-1301   PROD-368   73.12     Debit Card  completed   
4      TXN-13396   CUST-1225    PROD-76  171.17     Debit Card  completed   

            timestamp  
0 2025-01-01 03:23:00  
1 2025-01-01 04:50:00  
2 2025-01-01 05:18:00  
3 2025-01-01 07:14:00  
4 2025-01-01 08:57:00  


**Step 2: Simulate OLTP-style operations:**

1. Look up a single transaction by its ID (simulate a point query).
2. Insert a new transaction (append a row).
3. Update the status of a transaction from "pending" to "completed."
4. Check that the transaction is valid: the amount must be positive and the status must be one of the allowed values (simulate a consistency check).

1. **Point Query: Look up a Transaction**

This simulates a customer service agent or a user dashboard fetching details for a specific transaction_id.

In [70]:
# Select a specific ID to search for
target_id = "TXN-12345"

# Efficiently locate the row
transaction_detail = df[df['transaction_id'] == target_id]
print(transaction_detail)

     transaction_id customer_id product_id  amount payment_method   status  \
2745      TXN-12345   CUST-1240   PROD-139   148.9     Debit Card  pending   

               timestamp  
2745 2025-07-14 05:48:00  


2. **Append: Insert a New Transaction**

In OLTP, new data is constantly flowing in. We'll create a new record and use pd.concat to add it to our log.

In [71]:
new_txn = pd.DataFrame([{
    'transaction_id': 'TXN-15000',
    'customer_id': 'CUST-9999',
    'product_id': 'PROD-777',
    'amount': 150.50,
    'payment_method': 'Credit Card',
    'status': 'pending',
    'timestamp': datetime.now()
}])

df = pd.concat([df, new_txn], ignore_index=True)

3. **Update: Change Status**
   
This simulates a payment gateway confirming a transaction. We locate the "pending" transaction by ID and flip its status to "completed."

In [73]:
# Update TXN-15000 from pending to completed
df.loc[df['transaction_id'] == 'TXN-15000', 'status'] = 'completed'

4. **Consistency Check: Validation Logic**
   
Integrity is vital in OLTP systems. This function ensures that the data being processed doesn't violate business rules.

In [74]:
def validate_transaction(row):
    allowed_statuses = {'completed', 'refunded', 'pending'}
    
    # Check if amount is positive
    is_amount_valid = row['amount'] > 0
    
    # Check if status is in the allowed list
    is_status_valid = row['status'] in allowed_statuses
    
    return is_amount_valid and is_status_valid

# Testing the check on our newest row
latest_txn = df.iloc[-1]
is_valid = validate_transaction(latest_txn)

print(f"Transaction {latest_txn['transaction_id']} Valid: {is_valid}")

Transaction TXN-15000 Valid: True


**Step 3: Simulate OLAP-style operations on the same data:**

1. Compute total revenue by payment method.
2. Find the average transaction amount by month.
3. Identify the top 10 customers by total spending.
4. Compute the refund rate (percentage of refunded transactions).

1. **Total Revenue by Payment Method**

In [75]:
revenue_by_method = df[df['status'] == 'completed'].groupby('payment_method')['amount'].sum()
print("Total Revenue by Payment Method:")
print(revenue_by_method.sort_values(ascending=False))

Total Revenue by Payment Method:
payment_method
Credit Card    274437.00
PayPal         273439.61
Debit Card     263542.29
Crypto         255784.14
Name: amount, dtype: float64


2. **Average Transaction Amount by Month**

To do this, we first ensure the timestamp is a datetime object, then group by the month.

In [77]:
# Extract month for grouping
df['month'] = df['timestamp'].dt.to_period('M')

avg_monthly_txn = df.groupby('month')['amount'].mean()
print("\nAverage Transaction Amount by Month:")
print(avg_monthly_txn)


Average Transaction Amount by Month:
month
2025-01    261.262393
2025-02    248.208720
2025-03    254.728155
2025-04    262.724791
2025-05    246.408753
2025-06    237.525394
2025-07    246.990808
2025-08    250.285900
2025-09    249.902010
2025-10    260.519950
2025-11    249.081178
2025-12    266.643863
2026-03    150.500000
Freq: M, Name: amount, dtype: float64


3. **Top 10 Customers by Total Spending**

Identifying "Whales" or VIP customers is a classic OLAP task for marketing teams.

In [79]:
top_10_customers = df[df['status'] == 'completed'].groupby('customer_id')['amount'].sum()
top_10_customers = top_10_customers.sort_values(ascending=False).head(10)

print("\nTop 10 Customers by Spending:")
print(top_10_customers)


Top 10 Customers by Spending:
customer_id
CUST-1207    2910.30
CUST-1330    2808.10
CUST-760     2751.51
CUST-1162    2482.41
CUST-326     2448.28
CUST-1035    2401.51
CUST-1188    2395.70
CUST-1356    2316.11
CUST-619     2295.64
CUST-1362    2278.17
Name: amount, dtype: float64


4. **Compute the Refund Rate**
   
The refund rate is a critical KPI (Key Performance Indicator) for assessing product quality or fraud levels.

In [80]:
total_count = len(df)
refund_count = len(df[df['status'] == 'refunded'])

refund_rate = (refund_count / total_count) * 100
print(f"\nRefund Rate: {refund_rate:.2f}%")


Refund Rate: 5.30%


**Step 4: Write a markdown cell explaining: Why would OLTP operations typically use row-major storage while OLAP operations use column-major storage? How does the access pattern of each workload type map to the memory layout that serves it best?**

The choice between row-major and column-major storage is fundamentally about minimizing "wasted" data movement between the disk/memory and the CPU.

1. **OLTP: The Row-Major Specialist**

**Access Pattern:** "Point" operations (Find, Insert, Update).

OLTP workloads usually deal with entire records at once. When a customer logs in to view their order, the system needs the ID, the amount, the status, and the timestamp all together.

- Memory Layout: In row-major storage, all data for a single row is stored in contiguous memory blocks.

- The Benefit: To fetch one transaction, the disk head moves to one location and reads the entire "chunk" of data. This is highly efficient for writes as well—to add a new transaction, you simply append one block of data to the end of the file.

2. **OLAP: The Column-Major Specialist**
   
**Access Pattern:** "Scanning" operations (Sum, Average, Filter).

OLAP workloads rarely care about a single row. Instead, they ask questions like "What was the total sum of the 'amount' column?" Out of 50 columns in a table, an OLAP query might only need two.

- Memory Layout: In column-major storage, all values for the amount column are stored together, followed by all values for status, etc.

- The Benefit: * I/O Efficiency: If you only need to sum the amount, the system ignores the customer_id, product_id, and timestamp columns entirely. It only loads the relevant "amount" block into memory.

    - Compression: Because data in a single column is of the same type (e.g., all prices or all dates), it compresses much better than a row of mixed data types.

# **Task 5: Choosing the Right Model**

For each of the following real-world scenarios, recommend a data model (relational, document, or graph) and a processing paradigm (OLTP or OLAP). Write 3-4 sentences justifying each choice.

1. A hospital patient record system where doctors need to quickly look up a patient's complete medical history.
2. A social network that needs to suggest "friends of friends" connections.
3. A financial reporting system that computes monthly revenue breakdowns across product categories and regions.
4. An IoT platform that receives sensor readings from 10,000 devices and needs to detect anomalies in real time.
5. A content management system where each article has a different set of metadata fields.

Selecting the right data model and processing paradigm is a balancing act between how data is written and how it needs to be queried. Here are the recommendations for your scenarios:

1. **Hospital Patient Record System**
   
- Model: Relational (SQL)

- Paradigm: OLTP

- Justification: Patient safety relies on strict data integrity and ACID compliance, which relational databases provide through predefined schemas and constraints. An OLTP approach is necessary because doctors require immediate, low-latency access to a specific patient's record ("point queries") during consultations. The structured nature of relational tables ensures that critical fields like blood type or allergies are never missing or malformed.

2. **Social Network Friend Recommendations**

- Model: Graph

- Paradigm: OLTP (with light Analytical components)

- Justification: Social networks are defined by relationships rather than isolated data points; a Graph model (using nodes and edges) excels at traversing deep "friends of friends" hierarchies without the massive performance hit of recursive SQL joins. While the final recommendation might feel like an analysis, the process of finding connections in real-time as a user browses is an OLTP operation. This model allows the system to treat the relationship as a first-class citizen, making pathfinding much faster.

3. **Financial Reporting System**
   
- Model: Relational (often Column-oriented)

- Paradigm: OLAP

- Justification: Financial reporting involves scanning millions of historical rows to calculate sums and averages, which is the definition of an OLAP workload. A Relational model is preferred for its ability to join complex tables (Regions, Products, Sales), but using a column-major storage engine within that model will significantly speed up the aggregation of "Revenue" fields. This setup prioritizes read-heavy throughput over the speed of individual transaction updates.

4. **IoT Sensor Anomaly Detection**
   
- Model: Document or Time-Series

- Paradigm: OLTP (Stream Processing)

- Justification: IoT platforms face a high volume of disparate writes that must be processed as they arrive, making OLTP (specifically stream-oriented) the best fit for real-time detection. A Document model (or a specialized Time-Series database) is ideal because it handles the high-velocity ingestion of JSON-like packets from 10,000 different devices without requiring a rigid schema for every sensor type. This allows the system to flag an anomaly the second a reading deviates from the norm.

5. **Content Management System (CMS)**

- Model: Document (NoSQL)

- Paradigm: OLTP

- Justification: Since articles have varying metadata (e.g., a "Recipe" article needs ingredients, while a "Tech" article needs code snippets), a Document model is the most flexible choice. It stores each article as a self-contained object (like JSON), allowing for a "schema-on-read" approach where fields can differ from one document to the next. This is an OLTP environment because the primary goal is the rapid retrieval and editing of individual content pieces by authors and readers.